# Beer-Lambert Depth Estimation (Relative Seafloor Height)

Estimates local seafloor height variations from pseudo-reflectance using Beer-Lambert attenuation law.

**Physics**: Underwater light follows exponential attenuation → bright pixels = shallow/elevated (bombs), dark pixels = deep (pits)

**Workflow**:
1. Load transect and apply illumination correction
2. Compute Beer-Lambert depth: `cube.compute_beer_lambert_depth()`
3. Visualize heatmap: `cube.plot_georef(depth_overlay=True)`

In [ ]:
import importlib
import sys
from pathlib import Path

# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)
from utils.uhi.georef import *

In [ ]:
# Quick verification: Check that depth methods are loaded
print("✓ Loaded georef from:", georef.__file__)
print(
    "✓ compute_beer_lambert_depth available:",
    hasattr(georef.CombinedTransectCube, "compute_beer_lambert_depth"),
)

import inspect

print(
    "✓ plot_georef has depth_overlay parameter:",
    "depth_overlay"
    in inspect.signature(georef.CombinedTransectCube.plot_georef).parameters,
)

## Step 1: Load Transect 057

In [ ]:
# Load transect 057
transect = load_transect(config.TRANSECT_057_OUTPUT)
transect.list_files()

# Select file 5
cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

## Step 2: Apply Illumination Correction

**Important**: Beer-Lambert depth estimation requires illumination-corrected data

In [ ]:
# Apply illumination correction with 500px window (default)
cube.apply_illumination_correction_v2(window_size=500)

## Step 3: Compute Beer-Lambert Depth

Calculate relative seafloor height from attenuation

In [ ]:
# Compute Beer-Lambert depth estimation
cube.compute_beer_lambert_depth(
    track_start=config.UHI_TRACK_RANGE[0],
    track_end=config.UHI_TRACK_RANGE[1],
    window_size=500,  # Rolling mean window (same as illumination correction)
    wavelength_range=(500, 650),  # Optical window with stable attenuation
    quiet=False,
)

## Step 4: Visualize Depth Heatmap

Display depth variations using `plot_georef` with depth overlay

In [ ]:
# Plot depth heatmap with all cartographic features
cube.plot_georef(
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE[0],
    track_end=config.UHI_TRACK_RANGE[1],
    apply_alignment_shift=True,
    # DEPTH OVERLAY MODE
    depth_overlay=True,  # Use depth instead of RGB
    depth_cmap="RdBu_r",  # Blue=elevated/shallow, Red=deep/pits (SWAPPED COLORS)
    depth_vmin=None,  # Auto (2nd percentile)
    depth_vmax=None,  # Auto (98th percentile)
    depth_cbar_fraction=0.006,  # Colorbar width (increase for wider bar)
    depth_cbar_pad=0.04,  # Colorbar padding (increase to move away from plot)
    depth_cbar_shrink=0.5,  # Colorbar height (0.0-1.0, decrease for shorter bar)
    # Cartographic features (inherited from plot_georef)
    figsize=(50, 20),
    vmin=-0.145,
    vmax=0.11,
    # add_scale_bar=True,
    # scale_bar_length_m=1.0,
    # scale_bar_position="lower left",
    # scale_bar_color="black",
    # scale_bar_fontsize=20,
    # scale_bar_linewidth=4,
    # add_north_arrow=True,
    # north_arrow_position="upper right",
    # north_arrow_size=0.1,
    # north_arrow_color="black",
    # north_arrow_linewidth=4,
    # north_arrow_head_size=40,
)

## Optional: Custom Depth Range

Manually set depth colormap range for better contrast

In [ ]:
# Plot with custom depth range (e.g., ±0.2 m around mean)
cube.plot_georef(
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE[0],
    track_end=config.UHI_TRACK_RANGE[1],
    apply_alignment_shift=True,
    depth_overlay=True,
    depth_cmap="RdBu_r",  # Blue=elevated, Red=deep
    depth_vmin=-0.2,  # Manual min (meters)
    depth_vmax=0.2,  # Manual max (meters)
    depth_cbar_fraction=0.08,  # Example: Wider colorbar
    depth_cbar_pad=0.06,  # Example: More spacing from plot
    depth_cbar_shrink=0.8,  # Example: Shorter colorbar (80% of axis height)
    figsize=(40, 10),
)

## Summary

✅ **Beer-Lambert depth estimation complete!**

**Key Results:**
- `cube.depth_map`: 2D array of mean relative depth (meters)
- `cube.depth_spectral`: 3D array of per-wavelength depth
- `cube.depth_wavelengths`: Wavelengths used (500-650 nm)
- `cube.depth_attenuation`: c(λ) coefficients

**Interpretation:**
- **Red/warm colors**: Shallow / elevated areas (bombs, mounds)
- **Blue/cool colors**: Deep / depressed areas (pits)
- **Typical range**: 0-0.5 meters
- **Similar to**: MBES residuals (local height variations)